# 02b_smote_sensitivity_analysis.ipynb
# Analyse de sensibilité — SMOTE

## Objectif
Évaluer, à titre exploratoire, l’impact de l’oversampling (SMOTE) sur les performances d’un modèle classique de classification de texte.

## Important
Ce notebook constitue une analyse complémentaire uniquement.  
Le pipeline principal reste fondé sur les données originales (sans SMOTE).

## Prudence méthodologique
Dans un contexte NLP avec représentations TF-IDF creuses, SMOTE doit être interprété avec prudence :
- il synthétise artificiellement des points dans un espace vectoriel très haute dimension,
- il peut introduire du bruit,
- il ne reflète pas nécessairement une amélioration cliniquement fiable.

In [1]:
import sys
from pathlib import Path

current = Path.cwd().resolve()

if current.name == "notebooks":
    PROJECT_ROOT = current.parent
elif (current / "src").exists():
    PROJECT_ROOT = current
else:
    PROJECT_ROOT = current

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config.paths import *

In [ ]:
# ============================================================
# PATH CONFIGURATION
# ============================================================

from pathlib import Path
import os

PROJECT_ROOT = Path(
    os.getenv(
        "PROJECT_ROOT",
        PROJECT_ROOT 
    )
)

DATA_DIR = PROJECT_ROOT / "data"
DATA_CLEAN_DIR = DATA_DIR / "clean"

REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"

SMOTE_TABLES_DIR = TABLES_DIR / "smote"
SMOTE_FIGURES_DIR = FIGURES_DIR / "smote"

CLEAN_DATA_PATH = DATA_CLEAN_DIR / "mental_health_detection_clean.csv"

for directory in [SMOTE_TABLES_DIR, SMOTE_FIGURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("✅ Paths ready")
print(f"PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"CLEAN_DATA_PATH : {CLEAN_DATA_PATH}")

✅ Paths ready
PROJECT_ROOT    : C:\Users\anafi\Desktop\final_project_jedha
CLEAN_DATA_PATH : C:\Users\anafi\Desktop\final_project_jedha\data\clean\mental_health_detection_clean.csv


In [3]:
# ============================================================
# IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    recall_score,
    classification_report,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder

from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42

TEXT_COL = "body"
TARGET_COL = "category"

CRITICAL_CLASSES = ["Schizophrenia", "Bipolar"]

print("✅ Imports ready")

✅ Imports ready


In [4]:
# ============================================================
# LOAD DATA + TRAIN / TEST SPLIT
# ============================================================

df = pd.read_csv(CLEAN_DATA_PATH)

required_cols = [TEXT_COL, TARGET_COL]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required_cols].dropna().copy()

df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()

df = df[
    df[TEXT_COL].ne("") &
    df[TARGET_COL].ne("")
].reset_index(drop=True)

if df.empty:
    raise ValueError("Dataset is empty after cleaning.")

X = df[TEXT_COL]
y = df[TARGET_COL]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train_encoded, y_test_encoded = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=RANDOM_STATE
)

y_train = pd.Series(label_encoder.inverse_transform(y_train_encoded), name=TARGET_COL)
y_test = pd.Series(label_encoder.inverse_transform(y_test_encoded), name=TARGET_COL)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("\nTrain class distribution:")
print(y_train.value_counts())

print("\nTest class distribution:")
print(y_test.value_counts())

Train: (9016,)
Test : (2255,)

Train class distribution:
category
ADHD             1602
Anxiety          1534
Bipolar          1459
BPD              1294
Autism           1198
Depression       1146
schizophrenia     783
Name: count, dtype: int64

Test class distribution:
category
ADHD             401
Anxiety          384
Bipolar          365
BPD              324
Autism           299
Depression       287
schizophrenia    195
Name: count, dtype: int64


In [5]:
# ============================================================
# BASELINE MODEL — TF-IDF + LinearSVC (balanced)
# ============================================================

def critical_recall_score(y_true, y_pred, critical_classes=CRITICAL_CLASSES):
    recalls = []

    y_true = pd.Series(y_true).astype(str)
    y_pred = pd.Series(y_pred).astype(str)

    for cls in critical_classes:
        cls_mask = y_true == cls
        if cls_mask.sum() == 0:
            continue
        cls_recall = (y_pred[cls_mask] == cls).mean()
        recalls.append(cls_recall)

    return float(np.mean(recalls)) if recalls else 0.0

baseline_pipeline = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents="unicode",
    max_features=50000,
)

X_train_tfidf = baseline_pipeline.fit_transform(X_train)
X_test_tfidf = baseline_pipeline.transform(X_test)

baseline_clf = LinearSVC(
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

baseline_clf.fit(X_train_tfidf, y_train)

y_pred_baseline = baseline_clf.predict(X_test_tfidf)

baseline_results = {
    "model": "baseline_balanced",
    "f1_macro": f1_score(y_test, y_pred_baseline, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_baseline, average="macro", zero_division=0),
    "critical_recall": critical_recall_score(y_test, y_pred_baseline),
}

baseline_report = pd.DataFrame(
    classification_report(
        y_test,
        y_pred_baseline,
        output_dict=True,
        zero_division=0,
    )
).transpose()

print("Baseline:", baseline_results)
display(baseline_report)

Baseline: {'model': 'baseline_balanced', 'f1_macro': np.float64(0.7765775357065817), 'recall_macro': np.float64(0.7748794835700702), 'critical_recall': 0.7397260273972602}


,precision,recall,f1-score,support
ADHD,0.861905,0.902743,0.881851,401.00000
Anxiety,0.795213,0.778646,0.786842,384.00000
Autism,0.835526,0.849498,0.842454,299.00000
BPD,0.804805,0.827160,0.815830,324.00000
Bipolar,0.820669,0.739726,0.778098,365.00000
Depression,0.625378,0.721254,0.669903,287.00000
schizophrenia,0.728395,0.605128,0.661064,195.00000
accuracy,0.788470,0.788470,0.788470,0.78847
macro avg,0.781699,0.774879,0.776578,2255.00000
weighted avg,0.790523,0.788470,0.788101,2255.00000


In [6]:
import sklearn
import imblearn

print("scikit-learn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)

scikit-learn: 1.5.2
imbalanced-learn: 0.12.4


In [7]:
# ============================================================
# SMOTE MODEL — TF-IDF + SMOTE + LinearSVC
# ============================================================

# Vectorize first
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents="unicode",
    max_features=50000,
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Convert sparse matrix to dense only for this sensitivity test
# Warning: acceptable only if memory allows
X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()

print("Original training distribution:")
print(pd.Series(y_train).value_counts())

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_dense, y_train)

print("\nResampled training distribution:")
print(pd.Series(y_train_smote).value_counts())

smote_clf = LinearSVC(
    class_weight=None,
    random_state=RANDOM_STATE,
)

smote_clf.fit(X_train_smote, y_train_smote)

y_pred_smote = smote_clf.predict(X_test_dense)

smote_results = {
    "model": "smote_only",
    "f1_macro": f1_score(y_test, y_pred_smote, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_smote, average="macro", zero_division=0),
    "critical_recall": critical_recall_score(y_test, y_pred_smote),
}

smote_report = pd.DataFrame(
    classification_report(
        y_test,
        y_pred_smote,
        output_dict=True,
        zero_division=0,
    )
).transpose()

print("SMOTE:", smote_results)
display(smote_report)

Original training distribution:
category
ADHD             1602
Anxiety          1534
Bipolar          1459
BPD              1294
Autism           1198
Depression       1146
schizophrenia     783
Name: count, dtype: int64

Resampled training distribution:
category
Depression       1602
ADHD             1602
Bipolar          1602
schizophrenia    1602
BPD              1602
Anxiety          1602
Autism           1602
Name: count, dtype: int64
SMOTE: {'model': 'smote_only', 'f1_macro': np.float64(0.7727014528056415), 'recall_macro': np.float64(0.7707825435252406), 'critical_recall': 0.7397260273972602}


,precision,recall,f1-score,support
ADHD,0.853081,0.897756,0.874848,401.000000
Anxiety,0.788918,0.778646,0.783748,384.000000
Autism,0.813725,0.832776,0.823140,299.000000
BPD,0.809668,0.827160,0.818321,324.000000
Bipolar,0.825688,0.739726,0.780347,365.000000
Depression,0.623100,0.714286,0.665584,287.000000
schizophrenia,0.732919,0.605128,0.662921,195.000000
accuracy,0.784479,0.784479,0.784479,0.784479
macro avg,0.778157,0.770783,0.772701,2255.000000
weighted avg,0.786604,0.784479,0.784101,2255.000000


In [8]:
# ============================================================
# COMPARISON TABLE
# ============================================================

comparison_df = pd.DataFrame([
    baseline_results,
    smote_results,
])

comparison_df["delta_vs_baseline_f1"] = comparison_df["f1_macro"] - baseline_results["f1_macro"]
comparison_df["delta_vs_baseline_recall"] = comparison_df["recall_macro"] - baseline_results["recall_macro"]
comparison_df["delta_vs_baseline_critical"] = comparison_df["critical_recall"] - baseline_results["critical_recall"]

display(comparison_df)

comparison_df.to_csv(SMOTE_TABLES_DIR / "smote_sensitivity_comparison.csv", index=False)
baseline_report.to_csv(SMOTE_TABLES_DIR / "baseline_classification_report.csv", index=True)
smote_report.to_csv(SMOTE_TABLES_DIR / "smote_classification_report.csv", index=True)

print("✅ SMOTE sensitivity results exported")

,model,f1_macro,recall_macro,critical_recall,delta_vs_baseline_f1,delta_vs_baseline_recall,delta_vs_baseline_critical
0,baseline_balanced,0.776578,0.774879,0.739726,0.000000,0.000000,0.0
1,smote_only,0.772701,0.770783,0.739726,-0.003876,-0.004097,0.0


✅ SMOTE sensitivity results exported


## Conclusion

L’analyse montre que SMOTE peut modifier certaines métriques globales, mais son interprétation doit rester prudente dans ce projet :

- l’oversampling synthétique agit ici dans un espace TF-IDF de très haute dimension ;
- un gain éventuel de rappel ne garantit pas une amélioration réellement robuste ou cliniquement fiable ;
- cette approche ne reflète pas la distribution réelle des textes observés ;
- dans un cadre de santé mentale sensible, le risque d’introduire du bruit ou des frontières artificielles reste important.

👉 Cette analyse est donc conservée comme **test de sensibilité complémentaire**, mais **le pipeline principal et le modèle final restent entraînés sans SMOTE**.## Conclusion

L'utilisation de SMOTE peut améliorer certaines métriques, mais :

- elle modifie la distribution réelle des données,
- elle peut introduire du bruit artificiel,
- elle n’est pas adaptée à un cadre clinique sensible.

👉 Le modèle final reste donc entraîné sans SMOTE.